# RAGAS Evaluation: answer_similarity + answer_correctness

Loads cached eval_data (from `query_rag_system.ipynb`) and runs answer quality metrics.
Metrics are submitted as separate jobs for fault isolation — `answer_similarity` (fast, embedding-only) runs first.

**Prerequisites:**
- `eval_data_health_wallet.json` generated by `query_rag_system.ipynb`

**Metrics:**
- `answer_similarity` — Semantic similarity between RAG answer and ground truth (embedding cosine)
- `answer_correctness` — Factual correctness of answer vs ground truth (F1 + semantic similarity)

**See also:** `ragas_eval_context_metrics.ipynb` for context_precision + context_recall.

## Step 0: Setup and Dependencies

In [1]:
!pip install -q llama-stack-client==0.4.2 rich pandas

In [2]:
import json
import time
from datetime import datetime

import pandas as pd
from llama_stack_client import LlamaStackClient
from rich.pretty import pprint


def compute_aggregated(score_result):
    """Compute mean from per-question scores, skipping None/NaN entries.
    Falls back to RAGAS aggregated_results if available."""
    agg = score_result.aggregated_results
    if agg is not None and not (isinstance(agg, dict) and None in agg.values()):
        if isinstance(agg, dict):
            vals = [v for v in agg.values() if v is not None]
            return vals[0] if len(vals) == 1 else agg
        return agg
    scores = []
    for row in score_result.score_rows:
        s = row.get("score")
        if s is not None and str(s) != "nan":
            scores.append(float(s))
    return round(sum(scores) / len(scores), 6) if scores else None

In [3]:
# --- Configuration ---
RAGAS_URL = "http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321"
PROVIDER_ID_INLINE = "trustyai_ragas_inline"

In [4]:
ragas_client = LlamaStackClient(base_url=RAGAS_URL)

print("RAGAS system models:")
ragas_models = ragas_client.models.list()
for m in ragas_models:
    mid = getattr(m, 'identifier', None) or getattr(m, 'id', '?')
    mtype = getattr(m, 'model_type', '?')
    if hasattr(m, 'custom_metadata') and m.custom_metadata:
        mtype = m.custom_metadata.get('model_type', mtype)
    print(f"  {mid} ({mtype})")

print("\nRAGAS eval providers:")
providers = ragas_client.providers.list()
eval_providers = [p for p in providers if p.api == "eval"]
pprint(eval_providers)

RAGAS system models:


INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1/models "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1/providers "HTTP/1.1 200 OK"


  vllm-embedding/qwen3-4b-embedding (embedding)
  vllm-inference/Gemma-3-27B-BF16-Distributed (llm)

RAGAS eval providers:


[
│   ProviderInfo(
│   │   api='eval',
│   │   config={
│   │   │   'use_k8s': True,
│   │   │   'base_url': 'https://gemma-3-27b-bf16-distributed-vszp.apps.cluster-5pzpt.5pzpt.sandbox1134.opentlc.com/v1'
│   │   },
│   │   health={'status': 'Not Implemented', 'message': 'Provider does not implement health check'},
│   │   provider_id='trustyai_lmeval',
│   │   provider_type='remote::trustyai_lmeval'
│   ),
│   ProviderInfo(
│   │   api='eval',
│   │   config={
│   │   │   'embedding_model': 'vllm-embedding/qwen3-4b-embedding',
│   │   │   'ragas_config': {'raise_exceptions': False}
│   │   },
│   │   health={'status': 'Not Implemented', 'message': 'Provider does not implement health check'},
│   │   provider_id='trustyai_ragas_inline',
│   │   provider_type='inline::trustyai_ragas'
│   )
]

## Load Eval Data

In [5]:
with open("eval_data_health_wallet.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)
print(f"Loaded {len(eval_data)} evaluation entries")

Loaded 182 evaluation entries


## Step 4: Run RAGAS Evaluation

In [6]:
SCORING_FUNCTIONS = [
    "answer_similarity",
    "answer_correctness",
]

# Find the LLM model ID in RAGAS system
ragas_llm_model = None
for m in ragas_models:
    mid = getattr(m, 'identifier', None) or getattr(m, 'id', None)
    mtype = getattr(m, 'model_type', '')
    if hasattr(m, 'custom_metadata') and m.custom_metadata:
        mtype = m.custom_metadata.get('model_type', mtype)
    if mtype == 'llm':
        ragas_llm_model = mid
        break

assert ragas_llm_model, "No LLM model found in RAGAS system!"
print(f"Using LLM for evaluation: {ragas_llm_model}")


def run_ragas_metric(metric_names, eval_data, label=None):
    """Register dataset + benchmark, run eval, return results or None on failure."""
    label = label or "_".join(metric_names)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    ds_id = f"hw_{label}_{ts}"
    bm_id = f"hw_bench_{label}_{ts}"

    ragas_client.beta.datasets.register(
        dataset_id=ds_id,
        purpose="eval/question-answer",
        source={"type": "rows", "rows": eval_data},
        metadata={"provider_id": "localfs"},
    )
    ragas_client.alpha.benchmarks.register(
        benchmark_id=bm_id,
        dataset_id=ds_id,
        scoring_functions=metric_names,
        provider_id=PROVIDER_ID_INLINE,
    )

    job = ragas_client.alpha.eval.run_eval(
        benchmark_id=bm_id,
        benchmark_config={
            "eval_candidate": {
                "type": "model",
                "model": ragas_llm_model,
                "sampling_params": {"temperature": 0.1, "max_tokens": 500},
            },
            "scoring_params": {},
        },
    )
    print(f"[{label}] Job {job.job_id} submitted. Metrics: {metric_names}")

    start = time.time()
    while True:
        st = ragas_client.alpha.eval.jobs.status(benchmark_id=bm_id, job_id=job.job_id)
        elapsed = time.time() - start
        print(f"  [{elapsed:.0f}s] {st.status}")
        if st.status in ("completed", "failed"):
            break
        time.sleep(15)

    if st.status == "failed":
        print(f"  FAILED after {elapsed:.0f}s")
        return None

    results = ragas_client.alpha.eval.jobs.retrieve(benchmark_id=bm_id, job_id=job.job_id)
    print(f"  Completed in {elapsed:.0f}s")
    for mn in metric_names:
        if mn in results.scores:
            agg = compute_aggregated(results.scores[mn])
            print(f"  {mn}: {agg}")
    return results

Using LLM for evaluation: vllm-inference/Gemma-3-27B-BF16-Distributed


In [7]:
# Job 1: answer_similarity (embedding-only, fast)
results_sim = run_ragas_metric(["answer_similarity"], eval_data, label="similarity")

/tmp/ipykernel_3843/2514553265.py:28: DeprecationWarning: deprecated
  ragas_client.beta.datasets.register(
INFO:httpx:HTTP Request: POST http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1beta/datasets "HTTP/1.1 200 OK"
/tmp/ipykernel_3843/2514553265.py:34: DeprecationWarning: deprecated
  ragas_client.alpha.benchmarks.register(
INFO:httpx:HTTP Request: POST http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/hw_bench_similarity_20260507_160339/jobs "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/hw_bench_similarity_20260507_160339/jobs/1 "HTTP/1.1 200 OK"


[similarity] Job 1 submitted. Metrics: ['answer_similarity']
  [0s] in_progress


INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/hw_bench_similarity_20260507_160339/jobs/1 "HTTP/1.1 200 OK"


  [15s] in_progress


INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/hw_bench_similarity_20260507_160339/jobs/1 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/hw_bench_similarity_20260507_160339/jobs/1/result "HTTP/1.1 200 OK"


  [30s] completed
  Completed in 30s


In [8]:
# Job 2: answer_correctness (LLM-based, slower)
results_corr = run_ragas_metric(["answer_correctness"], eval_data, label="correctness")

/tmp/ipykernel_3843/2514553265.py:28: DeprecationWarning: deprecated
  ragas_client.beta.datasets.register(
INFO:httpx:HTTP Request: POST http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1beta/datasets "HTTP/1.1 200 OK"
/tmp/ipykernel_3843/2514553265.py:34: DeprecationWarning: deprecated
  ragas_client.alpha.benchmarks.register(
INFO:httpx:HTTP Request: POST http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/hw_bench_correctness_20260507_160410/jobs "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/hw_bench_correctness_20260507_160410/jobs/2 "HTTP/1.1 200 OK"


[correctness] Job 2 submitted. Metrics: ['answer_correctness']
  [0s] in_progress


INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/hw_bench_correctness_20260507_160410/jobs/2 "HTTP/1.1 200 OK"


  [15s] in_progress


INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/hw_bench_correctness_20260507_160410/jobs/2 "HTTP/1.1 200 OK"


...

  [5511s] in_progress


INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/hw_bench_correctness_20260507_160410/jobs/2 "HTTP/1.1 200 OK"


  [5526s] in_progress


INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/hw_bench_correctness_20260507_160410/jobs/2 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/hw_bench_correctness_20260507_160410/jobs/2/result "HTTP/1.1 200 OK"


  [5541s] completed
  Completed in 5541s
  answer_correctness: 0.873279


## Step 5: Results Analysis

In [9]:
# Merge scores from both jobs
all_scores = {}
ref_results = None
for r in [results_sim, results_corr]:
    if r is not None:
        ref_results = ref_results or r
        for mn, sr in r.scores.items():
            all_scores[mn] = sr

# Build results DataFrame
rows = []
if ref_results:
    for i, gen in enumerate(ref_results.generations):
        row = {"question": gen["user_input"][:80]}
        for metric in SCORING_FUNCTIONS:
            if metric in all_scores:
                score_rows = all_scores[metric].score_rows
                if i < len(score_rows):
                    score = score_rows[i].get("score", None)
                    if score is not None and str(score) != "nan":
                        row[metric] = score
                    else:
                        row[metric] = None
        rows.append(row)

df = pd.DataFrame(rows)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Summary
print("=" * 80)
print("RAGAS EVALUATION RESULTS - Health Wallet RAG")
print("=" * 80)
print(f"\nEval Data:    {len(eval_data)} questions")
print(f"Timestamp:    {timestamp}")
print()

print("AGGREGATED SCORES:")
print("-" * 40)
for metric in SCORING_FUNCTIONS:
    if metric in all_scores:
        sr = all_scores[metric]
        agg = compute_aggregated(sr)
        scored = sum(1 for r in sr.score_rows if r.get("score") is not None and str(r.get("score")) != "nan")
        skipped = len(sr.score_rows) - scored
        suffix = f" ({skipped} skipped)" if skipped > 0 else ""
        print(f"  {metric:25s}: {agg}{suffix}")
    else:
        print(f"  {metric:25s}: FAILED")
print()

RAGAS EVALUATION RESULTS - Health Wallet RAG

Eval Data:    182 questions
Timestamp:    20260507_173631

AGGREGATED SCORES:
----------------------------------------
  answer_similarity        : FAILED
  answer_correctness       : 0.873279 (19 skipped)



In [10]:
# Per-question breakdown
print("PER-QUESTION SCORES:")
print("-" * 80)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)
print(df.to_string(index=False))

PER-QUESTION SCORES:
--------------------------------------------------------------------------------
                                                                         question  answer_correctness
                                    Kde nájdem Peňaženku zdravia? Je spoplatnená?            0.612736
                         Prečo je Peňaženka zdravia viazaná na mobilnú aplikáciu?            0.550000
Na čerpanie príspevkov z Peňaženky zdravia musím mať mobilnú aplikáciu,\nalebo mi            0.748591
Ak som sa opäť vrátil do VšZP, môžem pre Peňaženku zdravia využívať svoje\nstaré             1.000000
      Som váš dlhoročný poistenec, prečo aj ja nemám nárok na Peňaženku\nzdravia?            0.487074
Aktualizácia mobilnej aplikácie prebehne automaticky, alebo si novú verziu\nmusím            0.997117
Máte kontakt na podporu pre klientov? Mám problémy s inštaláciou mobilnej\napliká            0.998846
 Prečo je potrebná aktivácia mobilnej aplikácie? Nie je to zbytočná strata\nčasu? 

In [11]:
# Identify weak spots
print("\nDIAGNOSTICS:")
print("=" * 60)

for metric in SCORING_FUNCTIONS:
    if metric not in df.columns:
        continue
    valid = df[df[metric].notna()]
    skipped = len(df) - len(valid)
    if skipped > 0:
        print(f"\n{metric}: {skipped}/{len(df)} entries skipped (parsing failure)")

    low_scores = valid[valid[metric] < 0.5]
    if len(low_scores) > 0:
        print(f"\n{metric} < 0.5 ({len(low_scores)} questions):")
        for _, row in low_scores.iterrows():
            print(f"  - {row['question']} (score: {row[metric]:.3f})")

# Summary interpretation
print("\n" + "=" * 60)
print("INTERPRETATION:")
for metric in SCORING_FUNCTIONS:
    if metric not in df.columns:
        continue
    valid = df[metric].dropna()
    if len(valid) == 0:
        print(f"  {metric:25s}: NO DATA")
        continue
    avg = valid.mean()
    if avg >= 0.8:
        verdict = "GOOD"
    elif avg >= 0.5:
        verdict = "NEEDS IMPROVEMENT"
    else:
        verdict = "POOR"
    print(f"  {metric:25s}: {avg:.3f} - {verdict}")


DIAGNOSTICS:

answer_correctness: 19/182 entries skipped (parsing failure)

answer_correctness < 0.5 (8 questions):
  - Som váš dlhoročný poistenec, prečo aj ja nemám nárok na Peňaženku
zdravia? (score: 0.487)
  - Budú sa finančné príspevky časom meniť? (score: 0.165)
  - Kde nájdem zmluvných zubárov, u ktorých môžem absolvovať dentálnu hygienu? (score: 0.471)
  - Dokedy si môžem vytvoriť skupinu v Peňaženke zdravia MAXI? (score: 0.407)
  - Je Peňaženka zdravia MAXI iba pre poistencov? (score: 0.189)
  - Ako požiadam o čerpanie finančných príspevkov? (score: 0.490)
  - Kde nájdem zmluvných zubárov, u ktorých môžem absolvovať dentálnu hygienu? (score: 0.471)
  - Ak je suma na doklade za dentálnu hygienu vyššia ako 40 €, môžem ju rozdeliť do  (score: 0.181)

INTERPRETATION:
  answer_correctness       : 0.873 - GOOD


## Save Results

Save experiment config and metric results to `results/` for cross-experiment comparison.

In [12]:
import os

os.makedirs("results", exist_ok=True)

# --- Experiment config ---
experiment_config = {
    "experiment_id": "experiment1",
    "description": "Baseline: Gemma-3-27B with default prompt",
    "timestamp": datetime.now().isoformat(),
    "eval_model": ragas_llm_model,
    "num_questions": len(eval_data),
}

with open("results/experiment_config.json", "w", encoding="utf-8") as f:
    json.dump(experiment_config, f, ensure_ascii=False, indent=2)
print("Saved results/experiment_config.json")

# --- Save each metric ---
for metric_name in SCORING_FUNCTIONS:
    m_result = {"metric": metric_name, "aggregated": None, "num_scored": 0, "num_skipped": 0, "per_question": []}
    if metric_name in all_scores:
        sr = all_scores[metric_name]
        m_result["aggregated"] = compute_aggregated(sr)
        for i, gen in enumerate(ref_results.generations):
            score = sr.score_rows[i].get("score", None) if i < len(sr.score_rows) else None
            is_valid = score is not None and str(score) != "nan"
            if is_valid:
                m_result["num_scored"] += 1
            else:
                m_result["num_skipped"] += 1
            entry = {"question": gen["user_input"], "score": round(float(score), 4) if is_valid else None}
            entry["response"] = eval_data[i]["response"] if i < len(eval_data) else ""
            entry["reference"] = eval_data[i]["reference"] if i < len(eval_data) else ""
            m_result["per_question"].append(entry)

    filename = f"results/{metric_name}.json"
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(m_result, f, ensure_ascii=False, indent=2)
    print(f"Saved {filename} (mean={m_result['aggregated']}, scored={m_result['num_scored']}, skipped={m_result['num_skipped']})")

print("\nAll results saved to results/ directory.")

Saved results/experiment_config.json
Saved results/answer_similarity.json (mean=None, scored=0, skipped=0)
Saved results/answer_correctness.json (mean=0.873279, scored=163, skipped=19)

All results saved to results/ directory.


In [17]:
  import os, json                                                                                                                                                                     
                                                                                                                                                                                      
  sr = results_sim.scores["semantic_similarity"]                                                                                                                                      
  agg = compute_aggregated(sr)                                                                                                                                                        
  m_result = {"metric": "answer_similarity", "aggregated": agg, "num_scored": 0, "num_skipped": 0, "per_question": []}                                                                
  for i, gen in enumerate(results_sim.generations):                                                                                                                                   
      score = sr.score_rows[i].get("score", None) if i < len(sr.score_rows) else None                                                                                                 
      is_valid = score is not None and str(score) != "nan"                                                                                                                            
      if is_valid:                                          
          m_result["num_scored"] += 1                                                                                                                                                 
      else:                                                 
          m_result["num_skipped"] += 1                                                                                                                                                
      m_result["per_question"].append({                     
          "question": gen["user_input"],                                                                                                                                              
          "score": round(float(score), 4) if is_valid else None,
          "response": eval_data[i]["response"] if i < len(eval_data) else "",                                                                                                         
          "reference": eval_data[i]["reference"] if i < len(eval_data) else "",                                                                                                       
      })                                                                                                                                                                              
                                                                                                                                                                                      
  os.makedirs("results", exist_ok=True)                                                                                                                                               
  with open("results/answer_similarity.json", "w", encoding="utf-8") as f:
      json.dump(m_result, f, ensure_ascii=False, indent=2)                                                                                                                            
  print(f"Saved results/answer_similarity.json (mean={agg}, scored={m_result['num_scored']}, skipped={m_result['num_skipped']})")

Saved results/answer_similarity.json (mean=0.9547265214319526, scored=182, skipped=0)


In [16]:
print(results_sim.scores.keys())

dict_keys(['semantic_similarity'])


## Auto-trigger: Context Metrics

Automatically runs `ragas_eval_context_metrics.ipynb` in a separate kernel. Results saved to the executed copy for review.

In [13]:
!jupyter nbconvert --to notebook --execute ragas_eval_context_metrics.ipynb \
    --output ragas_eval_context_metrics_executed.ipynb \
    --ExecutePreprocessor.timeout=28800
print("\nContext metrics notebook finished. Check ragas_eval_context_metrics_executed.ipynb for output.")

[NbConvertApp] Converting notebook ragas_eval_context_metrics.ipynb to notebook
[NbConvertApp] Writing 210910 bytes to ragas_eval_context_metrics_executed.ipynb

Context metrics notebook finished. Check ragas_eval_context_metrics_executed.ipynb for output.
